# Painting Generation with a GAN (WikiArt)

Starter notebook for the assignment:
1. Implement a GAN (DCGAN, 64×64)
2. Train on the [WikiArt dataset](https://www.kaggle.com/datasets/steubk/wikiart)
3. Generate paintings
4. Evaluate with **Inception Score** and **FID** following the
   [pytorch-ignite GAN evaluation blog](https://pytorch-ignite.ai/blog/gan-evaluation-with-fid-and-is/)
5. Collect everything needed for the 2-page NeurIPS report (hyperparameters, loss curves, IS/FID, sample grids)

**Tip:** Run this on a GPU (Kaggle Notebooks or Colab). On Kaggle you can just click *Add Data → steubk/wikiart*.

## 0. Setup

In [ ]:
!pip install -q pytorch-ignite scipy
# Only needed if you are NOT on Kaggle and want to download the dataset with kagglehub:
!pip install -q kagglehub

In [ ]:
import os, random, time, json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageFile

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torchvision.utils as vutils

from ignite.engine import Engine
from ignite.metrics import FID, InceptionScore

ImageFile.LOAD_TRUNCATED_IMAGES = True   # a few WikiArt files are truncated
Image.MAX_IMAGE_PIXELS = None            # some paintings are very large

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type != "cuda":
    print("WARNING: no GPU found. On Kaggle: Settings -> Accelerator -> GPU. Training on CPU will look 'stuck'.")
torch.backends.cudnn.benchmark = True

## 1. Hyperparameters
Keep everything in one dict so you can paste it straight into the report's hyperparameter table.

In [ ]:
cfg = dict(
    seed        = 42,
    image_size  = 64,       # DCGAN resolution
    nc          = 3,        # image channels
    nz          = 100,      # latent dim
    ngf         = 64,       # generator feature maps
    ndf         = 64,       # discriminator feature maps
    batch_size  = 128,
    num_epochs  = 25,
    lr_g        = 2e-4,
    lr_d        = 2e-4,
    beta1       = 0.5,      # Adam beta1 (DCGAN paper)
    beta2       = 0.999,
    label_smooth= 0.9,      # one-sided label smoothing for real labels (1.0 = off)
    style       = None,     # e.g. "Impressionism" to train on one style; None = all styles
    max_images  = 30000,    # cap dataset size for faster experiments (None = use all ~80k)
    num_workers = 4,        # only used for the one-time preprocessing pass (Kaggle has 4 CPUs)
    eval_samples= 5000,     # number of images used for FID/IS (more = more reliable, slower)
    out_dir     = "outputs",
)
os.makedirs(cfg["out_dir"], exist_ok=True)

random.seed(cfg["seed"]); np.random.seed(cfg["seed"]); torch.manual_seed(cfg["seed"])
print(json.dumps(cfg, indent=2))

## 2. Data
The dataset is organised as `wikiart/<Style>/<image>.jpg`, so `ImageFolder` works directly
(the style name becomes the class label, which a vanilla GAN ignores).

In [ ]:
# --- Locate the dataset -------------------------------------------------------
KAGGLE_PATH = Path("/kaggle/input/wikiart")
if KAGGLE_PATH.exists():
    data_root = KAGGLE_PATH
else:
    import kagglehub   # requires ~/.kaggle/kaggle.json or KAGGLE_USERNAME/KAGGLE_KEY env vars
    data_root = Path(kagglehub.dataset_download("steubk/wikiart"))

# The image folders may be nested one level down; find the folder containing style dirs
candidates = [data_root] + [p for p in data_root.iterdir() if p.is_dir()]
img_root = next(p for p in candidates
                if any(q.is_dir() and any(q.glob("*.jpg")) for q in p.iterdir() if q.is_dir()))
print("Image root:", img_root)
print("Styles:", sorted(d.name for d in img_root.iterdir() if d.is_dir()))

In [ ]:
transform = transforms.Compose([
    transforms.Resize(cfg["image_size"]),
    transforms.CenterCrop(cfg["image_size"]),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),   # -> [-1, 1] to match Tanh output
])

def safe_loader(path):
    # WikiArt images are often several thousand pixels wide. Decoding them at full size is the
    # main reason training is slow / appears stuck. img.draft() asks the JPEG decoder for a
    # reduced-size version directly (much faster); Resize then finishes the job.
    img = Image.open(path)
    img.draft("RGB", (cfg["image_size"] * 2, cfg["image_size"] * 2))
    return img.convert("RGB")

full_ds = dset.ImageFolder(root=str(img_root), transform=transform, loader=safe_loader)

idx = list(range(len(full_ds)))
if cfg["style"] is not None:
    cls = full_ds.class_to_idx[cfg["style"]]
    idx = [i for i, (_, y) in enumerate(full_ds.samples) if y == cls]
random.shuffle(idx)
if cfg["max_images"] is not None:
    idx = idx[: cfg["max_images"]]
dataset = Subset(full_ds, idx)
print(f"Selected images: {len(dataset)}")

### Preprocess once and cache
Decoding JPEGs every epoch is far slower than the GAN itself. We decode + resize **once**, store all images
as a single uint8 tensor (30k × 3 × 64 × 64 ≈ 370 MB), and train from memory. The cache is saved to
`outputs/`, so re-running the notebook in the same session skips this step. First pass: a few minutes.

In [ ]:
tag = f"{cfg['style'] or 'all'}_{cfg['max_images']}_{cfg['image_size']}_s{cfg['seed']}"
cache_path = f"{cfg['out_dir']}/wikiart_cache_{tag}.pt"

if os.path.exists(cache_path):
    images_u8 = torch.load(cache_path)
    print("Loaded cache:", cache_path)
else:
    pre_loader = DataLoader(dataset, batch_size=256, shuffle=False, num_workers=cfg["num_workers"])
    chunks, t0 = [], time.time()
    for k, (x, _) in enumerate(pre_loader):
        chunks.append(((x + 1) * 127.5).round().clamp(0, 255).to(torch.uint8))   # [-1,1] -> uint8
        if k % 10 == 0:
            print(f"  preprocessed {min((k + 1) * 256, len(dataset))}/{len(dataset)}  ({time.time()-t0:.0f}s)")
    images_u8 = torch.cat(chunks)
    torch.save(images_u8, cache_path)
    print(f"Saved cache ({images_u8.numel()/1e6:.0f} MB) in {time.time()-t0:.0f}s")

class CachedImages(torch.utils.data.Dataset):
    """Serves cached uint8 images as float tensors in [-1, 1] with random horizontal flips."""
    def __init__(self, x_u8, flip=True):
        self.x, self.flip = x_u8, flip
    def __len__(self):
        return len(self.x)
    def __getitem__(self, i):
        img = self.x[i].float() / 127.5 - 1.0
        if self.flip and random.random() < 0.5:
            img = img.flip(-1)
        return img, 0

train_ds = CachedImages(images_u8)
# num_workers=0: data is already in memory, and it avoids the DataLoader worker hangs seen on Kaggle
dataloader = DataLoader(train_ds, batch_size=cfg["batch_size"], shuffle=True,
                        num_workers=0, drop_last=True, pin_memory=(device.type == "cuda"))
print(f"Training images: {len(train_ds)}  |  batches/epoch: {len(dataloader)}")

In [ ]:
def show_grid(batch, title, nrow=8, save=None):
    grid = vutils.make_grid(batch[:64].cpu(), nrow=nrow, padding=2, normalize=True)
    plt.figure(figsize=(8, 8)); plt.axis("off"); plt.title(title)
    plt.imshow(np.transpose(grid, (1, 2, 0)))
    if save: plt.savefig(save, bbox_inches="tight", dpi=150)
    plt.show()

real_batch = next(iter(dataloader))
show_grid(real_batch[0], "Real WikiArt images", save=f"{cfg['out_dir']}/real_samples.png")

## 3. Model: DCGAN
Generator: latent `z ∈ R^100` → transposed convolutions → 3×64×64 image (Tanh).
Discriminator: strided convolutions with LeakyReLU → single logit.
(Architecture from Radford et al., 2016.)

In [ ]:
def weights_init(m):
    name = m.__class__.__name__
    if "Conv" in name:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif "BatchNorm" in name:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

class Generator(nn.Module):
    def __init__(self, nz, ngf, nc):
        super().__init__()
        self.main = nn.Sequential(
            nn.ConvTranspose2d(nz, ngf * 8, 4, 1, 0, bias=False),     # 4x4
            nn.BatchNorm2d(ngf * 8), nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),# 8x8
            nn.BatchNorm2d(ngf * 4), nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),# 16x16
            nn.BatchNorm2d(ngf * 2), nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),    # 32x32
            nn.BatchNorm2d(ngf), nn.ReLU(True),
            nn.ConvTranspose2d(ngf, nc, 4, 2, 1, bias=False),         # 64x64
            nn.Tanh(),
        )
    def forward(self, z):
        return self.main(z)

class Discriminator(nn.Module):
    def __init__(self, ndf, nc):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(nc, ndf, 4, 2, 1, bias=False),                  # 32x32
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),             # 16x16
            nn.BatchNorm2d(ndf * 2), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),         # 8x8
            nn.BatchNorm2d(ndf * 4), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False),         # 4x4
            nn.BatchNorm2d(ndf * 8), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 8, 1, 4, 1, 0, bias=False),               # 1x1 logit
        )
    def forward(self, x):
        return self.main(x).view(-1)

netG = Generator(cfg["nz"], cfg["ngf"], cfg["nc"]).to(device); netG.apply(weights_init)
netD = Discriminator(cfg["ndf"], cfg["nc"]).to(device);        netD.apply(weights_init)

count = lambda m: sum(p.numel() for p in m.parameters())
print(f"G params: {count(netG)/1e6:.2f}M | D params: {count(netD)/1e6:.2f}M")

## 4. Training
Standard non-saturating GAN loss with `BCEWithLogitsLoss` (numerically stabler than Sigmoid + BCELoss).
We log per-iteration losses and D(x), D(G(z)), and save a fixed-noise sample grid each epoch.

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optD = optim.Adam(netD.parameters(), lr=cfg["lr_d"], betas=(cfg["beta1"], cfg["beta2"]))
optG = optim.Adam(netG.parameters(), lr=cfg["lr_g"], betas=(cfg["beta1"], cfg["beta2"]))

fixed_noise = torch.randn(64, cfg["nz"], 1, 1, device=device)
history = {"G": [], "D": [], "D_x": [], "D_G_z": []}
epoch_grids = []

In [ ]:
start = time.time()
for epoch in range(cfg["num_epochs"]):
    netG.train(); netD.train()
    t_epoch = time.time()
    for i, (real, _) in enumerate(dataloader):
        real = real.to(device, non_blocking=True)
        b = real.size(0)
        real_lbl = torch.full((b,), cfg["label_smooth"], device=device)
        fake_lbl = torch.zeros(b, device=device)

        # ---- (1) Update D: maximize log D(x) + log(1 - D(G(z))) ----
        netD.zero_grad(set_to_none=True)
        out_real = netD(real)
        loss_real = criterion(out_real, real_lbl)
        z = torch.randn(b, cfg["nz"], 1, 1, device=device)
        fake = netG(z)
        out_fake = netD(fake.detach())
        loss_fake = criterion(out_fake, fake_lbl)
        lossD = loss_real + loss_fake
        lossD.backward()
        optD.step()

        # ---- (2) Update G: maximize log D(G(z)) (non-saturating) ----
        netG.zero_grad(set_to_none=True)
        out = netD(fake)
        lossG = criterion(out, torch.ones(b, device=device))
        lossG.backward()
        optG.step()

        history["G"].append(lossG.item()); history["D"].append(lossD.item())
        history["D_x"].append(torch.sigmoid(out_real).mean().item())
        history["D_G_z"].append(torch.sigmoid(out_fake).mean().item())

        if i % 100 == 0:
            print(f"[{epoch+1}/{cfg['num_epochs']}][{i}/{len(dataloader)}] "
                  f"Loss_D {lossD.item():.3f}  Loss_G {lossG.item():.3f}  "
                  f"D(x) {history['D_x'][-1]:.3f}  D(G(z)) {history['D_G_z'][-1]:.3f}")

    # end of epoch: fixed-noise snapshot + checkpoint
    print(f"==> epoch {epoch+1} done in {time.time()-t_epoch:.0f}s")
    netG.eval()
    with torch.no_grad():
        snap = netG(fixed_noise).cpu()
    epoch_grids.append(vutils.make_grid(snap, nrow=8, padding=2, normalize=True))
    vutils.save_image(snap, f"{cfg['out_dir']}/epoch_{epoch+1:03d}.png", nrow=8, normalize=True)
    torch.save({"G": netG.state_dict(), "D": netD.state_dict(), "cfg": cfg, "epoch": epoch + 1},
               f"{cfg['out_dir']}/checkpoint.pt")

print(f"Training finished in {(time.time()-start)/60:.1f} min")
json.dump(history, open(f"{cfg['out_dir']}/history.json", "w"))

## 5. Training loss curves

In [ ]:
def smooth(x, k=50):
    x = np.asarray(x)
    return np.convolve(x, np.ones(k) / k, mode="valid") if len(x) > k else x

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["G"], alpha=0.25, color="C0"); axes[0].plot(smooth(history["G"]), label="Generator", color="C0")
axes[0].plot(history["D"], alpha=0.25, color="C1"); axes[0].plot(smooth(history["D"]), label="Discriminator", color="C1")
axes[0].set_xlabel("Iteration"); axes[0].set_ylabel("Loss"); axes[0].set_title("GAN losses"); axes[0].legend()

axes[1].plot(smooth(history["D_x"]), label="D(x)  (real)")
axes[1].plot(smooth(history["D_G_z"]), label="D(G(z))  (fake)")
axes[1].set_xlabel("Iteration"); axes[1].set_ylabel("Probability"); axes[1].set_title("Discriminator outputs"); axes[1].legend()
plt.tight_layout(); plt.savefig(f"{cfg['out_dir']}/loss_curves.png", dpi=200, bbox_inches="tight"); plt.show()

## 6. Generated paintings

In [ ]:
# Progression over training (first, middle, last epoch)
picks = sorted(set([0, len(epoch_grids) // 2, len(epoch_grids) - 1]))
fig, axes = plt.subplots(1, len(picks), figsize=(6 * len(picks), 6))
for ax, e in zip(np.atleast_1d(axes), picks):
    ax.imshow(np.transpose(epoch_grids[e], (1, 2, 0))); ax.axis("off"); ax.set_title(f"Epoch {e+1}")
plt.tight_layout(); plt.savefig(f"{cfg['out_dir']}/progression.png", dpi=150, bbox_inches="tight"); plt.show()

# Fresh samples from the final generator
netG.eval()
with torch.no_grad():
    samples = netG(torch.randn(64, cfg["nz"], 1, 1, device=device))
show_grid(samples, "Generated paintings (final model)", save=f"{cfg['out_dir']}/generated_samples.png")

In [ ]:
# Optional: latent-space interpolation between two random paintings (nice figure for the report)
with torch.no_grad():
    z0, z1 = torch.randn(2, cfg["nz"], 1, 1, device=device)
    alphas = torch.linspace(0, 1, 10, device=device).view(-1, 1, 1, 1)
    interp = netG((1 - alphas) * z0 + alphas * z1)
vutils.save_image(interp, f"{cfg['out_dir']}/interpolation.png", nrow=10, normalize=True)
show_grid(interp, "Latent interpolation", nrow=10)

## 7. Evaluation: Inception Score & FID (pytorch-ignite)
Following the ignite blog: build an evaluation `Engine` whose step returns `(fake, real)` batches
resized to 299×299 (InceptionV3 input size). `FID` uses both; `InceptionScore` only uses the fake images
(`output_transform=lambda x: x[0]`).

Note: our images are in [-1, 1], so we map them to [0, 1] before resizing.
Higher IS is better; lower FID is better. FID from 64×64 upsampled images will be high in absolute
terms — report the number of samples you used, since FID is biased by sample size.

In [ ]:
fid_metric = FID(device=device)
is_metric = InceptionScore(device=device, output_transform=lambda x: x[0])

import torch.nn.functional as F

def interpolate(batch):
    # batch: tensor in [0, 1], shape (B, 3, H, W) -> (B, 3, 299, 299).
    # The ignite blog resizes one image at a time with PIL on the CPU, which is very slow for
    # thousands of samples; this does the same bilinear resize for the whole batch on the GPU.
    return F.interpolate(batch.to(device), size=(299, 299), mode="bilinear", align_corners=False)

denorm = lambda x: (x.clamp(-1, 1) + 1) / 2

def evaluation_step(engine, batch):
    with torch.no_grad():
        real = batch[0].to(device)
        noise = torch.randn(real.size(0), cfg["nz"], 1, 1, device=device)
        netG.eval()
        fake = netG(noise)
        fake = interpolate(denorm(fake)).to(device)
        real = interpolate(denorm(real)).to(device)
        return fake, real

evaluator = Engine(evaluation_step)
fid_metric.attach(evaluator, "fid")
is_metric.attach(evaluator, "is")

# Use a held-out-sized subset of real images for evaluation
eval_idx = random.sample(range(len(images_u8)), min(cfg["eval_samples"], len(images_u8)))
eval_loader = DataLoader(Subset(CachedImages(images_u8, flip=False), eval_idx),
                         batch_size=64, shuffle=False, num_workers=0)

t0 = time.time()
evaluator.run(eval_loader, max_epochs=1)
fid_score = evaluator.state.metrics["fid"]
is_score = evaluator.state.metrics["is"]
print(f"FID: {fid_score:.2f}   IS: {is_score:.3f}   ({len(eval_idx)} samples, {time.time()-t0:.0f}s)")
json.dump({"fid": float(fid_score), "is": float(is_score), "n_samples": len(eval_idx)},
          open(f"{cfg['out_dir']}/metrics.json", "w"), indent=2)

In [ ]:
# (Optional) Sanity-check baseline: IS of *real* images at the same resolution gives an upper reference
is_real = InceptionScore(device=device, output_transform=lambda x: x[1])
real_eval = Engine(evaluation_step); is_real.attach(real_eval, "is_real")
real_eval.run(eval_loader, max_epochs=1)
print(f"IS of real 64x64 images: {real_eval.state.metrics['is_real']:.3f}")

## 8. Summary for the report
Run the cell below and copy the output into your NeurIPS LaTeX table. Figures saved in `outputs/`:
`loss_curves.png`, `generated_samples.png`, `progression.png`, `interpolation.png`, `real_samples.png`.

In [ ]:
print("\\begin{tabular}{ll}\n\\toprule\nHyperparameter & Value \\\\\n\\midrule")
for k in ["image_size", "nz", "ngf", "ndf", "batch_size", "num_epochs", "lr_g", "lr_d",
          "beta1", "beta2", "label_smooth", "style", "max_images"]:
    print(f"{k.replace('_', ' ')} & {cfg[k]} \\\\")
print("\\bottomrule\n\\end{tabular}")
print(f"\nFID = {fid_score:.2f}, IS = {is_score:.3f} (on {len(eval_idx)} samples)")

## 9. Ideas for experiments (to make the report interesting)
- **Single style vs. all styles**: set `cfg["style"] = "Impressionism"` (or `"Ukiyo_e"`, `"Cubism"`) — a narrower distribution usually gives lower FID.
- **Learning-rate balance**: e.g. TTUR (`lr_d = 4e-4`, `lr_g = 1e-4`).
- **Label smoothing on/off** (`label_smooth = 1.0` disables it).
- **Track FID over epochs**: wrap the evaluation cell in a function and call it every few epochs with ~1k samples to plot FID vs. epoch.
- **Stronger models** if time allows: spectral normalisation in D (`nn.utils.spectral_norm`), WGAN-GP loss, or a conditional GAN using the style label.